# AOI Rendering Pipeline

The renderer composes synthetic AOI images from three layers:

1. **Background** -- dark silicon die substrate with bond pads and traces
2. **Wires** -- bond wires with specular reflection (cylindrical model)
3. **Defect cues** -- subtle visual artifacts (shadows, discolouration)

## Optical model

| Surface | Model |
|---------|-------|
| Substrate | Diffuse (Lambertian) with texture noise |
| Bond pads | Flat metallic with slight noise |
| Wires | Specular highlight (Gaussian along centre) + diffuse falloff |
| Defects | Shadow (break), tint (lift) |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from udm_epic6.rendering.aoi_renderer import render_background, render_aoi_image
from udm_epic6.models.wire_geometry import generate_wire_profile

In [ ]:
# Render background only
rng = np.random.default_rng(42)
bg = render_background(256, 256, rng=rng)

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(bg)
ax.set_title("Die Surface Background")
ax.axis('off')
plt.show()

In [ ]:
# Full AOI image with wires
rng = np.random.default_rng(42)
profiles = generate_wire_profile(rng, image_size=(256, 256), n_wires=4)
img = render_aoi_image(profiles, height=256, width=256, rng=rng)

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(img)
ax.set_title("Full AOI Image (4 wires, no defects)")
ax.axis('off')
plt.show()

In [ ]:
# Grid of random samples from the dataset
from udm_epic6.data.dataset import BondWireDataset

ds = BondWireDataset(n_samples=8, image_size=(256, 256), seed=99)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(8):
    sample = ds[i]
    ax = axes[i // 4, i % 4]
    img = sample['image'].permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(f"{sample['defect_type']}\n(sev={sample['metadata']['severity']:.2f})",
                 fontsize=9)
    ax.axis('off')

fig.suptitle("Random Dataset Samples", fontsize=14)
fig.tight_layout()
plt.show()

In [ ]:
# Show image + mask side by side
sample = ds[3]
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(sample['image'].permute(1, 2, 0).numpy())
axes[0].set_title(f"AOI Image ({sample['defect_type']})")
axes[1].imshow(sample['mask'].squeeze(0).numpy(), cmap='hot')
axes[1].set_title("Defect Mask")
for ax in axes:
    ax.axis('off')
fig.tight_layout()
plt.show()